# Auto-downloader for latest NSPL and ONSPD

This script searches data.gov.uk each time it runs, instead of relying on one fixed package ID.

It supports:

- **NSPL** — National Statistics Postcode Lookup
- **ONSPD** — ONS Postcode Directory

## How the search works

It calls:

```text
https://data.gov.uk/api/action/package_search
```

It uses the `q`, `rows`, and `sort` parameters.

## Keys used from the API response

Dataset-level keys:

- `id`
- `name`
- `title`
- `notes`
- `metadata_modified`
- `organization.title`
- `organization.name`

Resource-level keys:

- `resources[].name`
- `resources[].description`
- `resources[].format`
- `resources[].url`

## References

- data.gov.uk API documentation: https://guidance.data.gov.uk/get_data/api_documentation/
- ONS postcode products: https://www.ons.gov.uk/methodology/geography/geographicalproducts/postcodeproducts
- Rolling NSPL dataset you found: https://www.data.gov.uk/dataset/7ec10db7-c8f4-4a40-8d82-8921935b4865/national-statistics-postcode-lookup-uk
- CKAN API guide - Most of the code is derived from this tutorial: https://docs.ckan.org/en/latest/api/
- CKAN GitHub Repo - This repo was very helpful for CKAN-style automation: https://github.com/ckan/ckanapi
- Another CKAN GitHub Repo - Another one that one useful for understanding CKAN resources: https://github.com/reubano/ckanutils

In [1]:
"""
download_latest_nspl_onspd_search_based.py

Purpose:
    Search data.gov.uk for the latest NSPL and ONSPD download records,
    find the CSV resource, and download it.

Products:
    NSPL  = National Statistics Postcode Lookup
    ONSPD = ONS Postcode Directory

Why search-based?
    The official ONS quarterly dataset IDs can change when a new release is published.
    So this script searches data.gov.uk each time instead of relying on one fixed ID.
    Previous versions of the scripts where not search based and might not adopt to newly published data

Important:
    ONSPD is very large. Test NSPL first if you are unsure.

References:
    data.gov.uk API documentation:
        https://guidance.data.gov.uk/get_data/api_documentation/

    CKAN API guide - Most of the code is derived from this tutorial:
        https://docs.ckan.org/en/latest/api/

    CKAN GitHub Repo - This repo was very helpful for CKAN-style automation:
        https://github.com/ckan/ckanapi
        
    Another CKAN GitHub Repo - Another one that one useful for understanding CKAN resources:
        https://github.com/reubano/ckanutils

    ONS postcode products:
        https://www.ons.gov.uk/methodology/geography/geographicalproducts/postcodeproducts

    Example rolling NSPL dataset:
        https://www.data.gov.uk/dataset/7ec10db7-c8f4-4a40-8d82-8921935b4865/national-statistics-postcode-lookup-uk

    Example official ONSPD hosted table:
        https://www.data.gov.uk/dataset/65a2ee49-d331-41c9-b09f-c7defc68b965/ons-postcode-directory-may-2026-for-the-united-kingdom-hosted-table

"""

from __future__ import annotations

import re
import traceback
from datetime import datetime
from pathlib import Path
from typing import Any, Dict, List, Optional, Tuple

import requests


# =============================================================================
# 1. Settings
# =============================================================================

# data.gov.uk is built on CKAN.
# CKAN calls datasets "packages".
DATA_GOV_PACKAGE_SEARCH_URL = "https://data.gov.uk/api/action/package_search"

# Where downloaded files will be saved.
OUTPUT_FOLDER = Path("ons_postcode_downloads")

# Choose which products to download.
# For a quick safe test, use only:
#     PRODUCTS_TO_DOWNLOAD = ["NSPL"]
#
# For both NSPL and ONSPD, use:
#     PRODUCTS_TO_DOWNLOAD = ["NSPL", "ONSPD"]
PRODUCTS_TO_DOWNLOAD = ["NSPL", "ONSPD"]

# We want CSV download resources.
PREFERRED_RESOURCE_FORMAT = "CSV"


PRODUCTS = {
    "NSPL": {
        "friendly_name": "National Statistics Postcode Lookup",
        # Search queries used for NSPL.
        # Query 1 catches official ONS hosted-table releases.
        # Query 2 catches the rolling dataset page you found.
        # Query 3 is a broader backup.
        "queries": [
            '"National Statistics Postcode Lookup" "Hosted Table"',
            '"National Statistics Postcode Lookup UK"',
            'NSPL "National Statistics Postcode Lookup"',
        ],
        "canonical_phrases": [
            "national statistics postcode lookup",
            "nspl",
        ],
        "strong_title_phrase": "national statistics postcode lookup",
        "allowed_publishers": [
            "office for national statistics",
            "london borough of camden",
        ],
        "reject_phrases": [
            "user guide",
            "centroid",
            "online ons postcode directory",
            "live postcodes",
        ],
        # For the rolling Camden record, there may be no month-year in the title.
        # We allow metadata_modified as a fallback date for NSPL.
        "allow_metadata_date_fallback": True,
    },
    "ONSPD": {
        "friendly_name": "ONS Postcode Directory",
        "queries": [
            '"ONS Postcode Directory" "Hosted Table"',
            'ONSPD "Hosted Table"',
            '"ONS Postcode Directory" "United Kingdom" "CSV"',
        ],
        "canonical_phrases": [
            "ons postcode directory",
            "onspd",
        ],
        "strong_title_phrase": "ons postcode directory",
        "allowed_publishers": [
            "office for national statistics",
        ],
        "reject_phrases": [
            "user guide",
            "centroid",
            "online ons postcode directory",
            "live postcodes",
        ],
        # For official ONSPD releases, prefer the date in the title:
        # e.g. "(May 2026)".
        "allow_metadata_date_fallback": False,
    },
}


MONTH_NAME_TO_NUMBER = {
    "january": 1,
    "february": 2,
    "march": 3,
    "april": 4,
    "may": 5,
    "june": 6,
    "july": 7,
    "august": 8,
    "september": 9,
    "october": 10,
    "november": 11,
    "december": 12,
}


# =============================================================================
# 2. Helper functions: dates and filenames
# =============================================================================

def safe_filename(text: str) -> str:
    """
    Make a safe Windows-friendly filename.

    Changes some symbols to underscores _.
    """
    text = text.strip()
    text = re.sub(r"[^A-Za-z0-9._-]+", "_", text)
    text = re.sub(r"_+", "_", text)
    return text.strip("_")


def parse_month_year_from_text(text: str) -> Optional[Tuple[int, int, int]]:
    """
    Find a date like "(May 2026)" in a dataset title.

    Returns:
        (year, month, day)

    We use day = 1 because the release title only gives month and year.

    Example:
        "ONS Postcode Directory (May 2026) for the UK (Hosted Table)"
        becomes:
        (2026, 5, 1)
    """
    pattern = (
        r"\("
        r"(January|February|March|April|May|June|July|August|September|October|November|December)"
        r"\s+"
        r"(\d{4})"
        r"\)"
    )

    match = re.search(pattern, text, flags=re.IGNORECASE)

    if match is None:
        return None

    month_name = match.group(1).lower()
    year = int(match.group(2))
    month = MONTH_NAME_TO_NUMBER[month_name]

    return year, month, 1


def parse_metadata_modified(value: str) -> Optional[Tuple[int, int, int]]:
    """
    Parse data.gov.uk metadata_modified.

    If the title has no release month, we can use the page's last-updated date.
    """
    if not value:
        return None

    # Common CKAN format looks like:
    # 2026-06-22T12:34:56.123456
    try:
        dt = datetime.fromisoformat(value.replace("Z", "+00:00"))
        return dt.year, dt.month, dt.day
    except ValueError:
        return None


# =============================================================================
# 3. Helper functions: data.gov.uk search
# =============================================================================

def call_package_search(query: str, rows: int = 100) -> List[Dict[str, Any]]:
    """
    Call data.gov.uk package_search.

    API route:
        https://data.gov.uk/api/action/package_search

    Search keys used:
        q     = free-text search query
        rows  = maximum number of results
        sort  = order from data.gov.uk

    Summary: We ask the data.gov.uk catalogue:
        "Show me datasets matching these words."
    """
    params = {
        "q": query,
        "rows": rows,
        "sort": "metadata_modified desc",
    }

    response = requests.get(
        DATA_GOV_PACKAGE_SEARCH_URL,
        params=params,
        timeout=60,
    )
    response.raise_for_status()

    data = response.json()

    if not data.get("success"):
        raise RuntimeError(f"data.gov.uk API returned success=false: {data}")

    return data["result"]["results"]


def dataset_text_blob(dataset: Dict[str, Any]) -> str:
    """
    Combine useful dataset fields into one lowercase text blob.

    Keys used from data.gov.uk / CKAN:
        id
        name
        title
        notes
        organization.title
        organization.name
        resources[].name
        resources[].description
        resources[].format
        resources[].url

    Summary:
        We make one big label from the dataset page
        so it is easier to check if it is the right dataset.
    """
    pieces = [
        str(dataset.get("id", "")),
        str(dataset.get("name", "")),
        str(dataset.get("title", "")),
        str(dataset.get("notes", "")),
    ]

    organisation = dataset.get("organization") or {}
    pieces.append(str(organisation.get("title", "")))
    pieces.append(str(organisation.get("name", "")))

    for resource in dataset.get("resources", []):
        pieces.append(str(resource.get("name", "")))
        pieces.append(str(resource.get("description", "")))
        pieces.append(str(resource.get("format", "")))
        pieces.append(str(resource.get("url", "")))

    return " ".join(pieces).lower()


def has_csv_resource(dataset: Dict[str, Any]) -> bool:
    """
    Check if a dataset has a CSV resource.

    Keys used:
        resources[].format
        resources[].url
        resources[].name
        resources[].description
    """
    for resource in dataset.get("resources", []):
        resource_format = str(resource.get("format", "")).upper().strip()
        resource_url = resource.get("url")

        if resource_format == "CSV" and resource_url:
            return True

    # Backup if the format field is messy.
    for resource in dataset.get("resources", []):
        text = " ".join(
            [
                str(resource.get("name", "")),
                str(resource.get("description", "")),
                str(resource.get("url", "")),
            ]
        ).lower()

        if "csv" in text and resource.get("url"):
            return True

    return False


def choose_csv_resource(dataset: Dict[str, Any]) -> Dict[str, Any]:
    """
    Return the best CSV resource from the dataset.
    """
    resources = dataset.get("resources", [])

    # First choice: exact CSV format.
    for resource in resources:
        resource_format = str(resource.get("format", "")).upper().strip()
        resource_url = resource.get("url")

        if resource_format == "CSV" and resource_url:
            return resource

    # Backup: CSV mentioned in name, description, or URL.
    for resource in resources:
        text = " ".join(
            [
                str(resource.get("name", "")),
                str(resource.get("description", "")),
                str(resource.get("url", "")),
            ]
        ).lower()

        if "csv" in text and resource.get("url"):
            return resource

    raise RuntimeError(f"No CSV resource found for dataset: {dataset.get('title')}")


def publisher_name(dataset: Dict[str, Any]) -> str:
    """
    Get publisher name from CKAN organization metadata.
    """
    organisation = dataset.get("organization") or {}
    return str(organisation.get("title") or organisation.get("name") or "").lower()


# =============================================================================
# 4. Candidate scoring
# =============================================================================

def score_dataset(dataset: Dict[str, Any], product_key: str, settings: Dict[str, Any]) -> Optional[Dict[str, Any]]:
    """
    Score one dataset.

    The script keeps the highest scoring newest candidate.

    Scoring idea:
        - Must look like the right product.
        - Must not be a user guide or centroid-only file.
        - Must have a CSV resource.
        - Prefer official ONS hosted tables.
        - Allow the rolling Camden NSPL record as a fallback/alternative.
        - I have found that Camden regularly releases an updated version and API host it as well.
    """
    title = str(dataset.get("title", ""))
    title_lower = title.lower()
    blob = dataset_text_blob(dataset)
    publisher = publisher_name(dataset)

    # Reject unwanted records.
    for phrase in settings["reject_phrases"]:
        if phrase in blob:
            return None

    # Must have at least one canonical phrase.
    if not any(phrase in blob for phrase in settings["canonical_phrases"]):
        return None

    # Must have a CSV resource.
    if not has_csv_resource(dataset):
        return None

    # If the publisher is present, it must be one of our expected publishers.
    # For NSPL, we allow Camden because of the rolling data.gov.uk page you found.
    if publisher:
        if not any(allowed in publisher for allowed in settings["allowed_publishers"]):
            return None

    # Get release date.
    release_date = parse_month_year_from_text(title)

    # If title has no date, optionally use metadata_modified.
    date_source = "title_month_year"

    if release_date is None and settings["allow_metadata_date_fallback"]:
        release_date = parse_metadata_modified(str(dataset.get("metadata_modified", "")))
        date_source = "metadata_modified"

    # For ONSPD, we require a release month in title because official releases are dated.
    if release_date is None:
        return None

    score = 0
    reasons = []

    if settings["strong_title_phrase"] in title_lower:
        score += 50
        reasons.append("canonical product phrase in title")

    if "hosted table" in title_lower:
        score += 30
        reasons.append("hosted table in title")

    if "office for national statistics" in publisher:
        score += 25
        reasons.append("publisher is Office for National Statistics")

    if product_key == "NSPL" and "london borough of camden" in publisher:
        score += 10
        reasons.append("rolling NSPL record from Camden/data.gov.uk")

    if has_csv_resource(dataset):
        score += 20
        reasons.append("has CSV resource")

    # Dated official release is clearer than metadata fallback.
    if date_source == "title_month_year":
        score += 10
        reasons.append("release date found in title")
    else:
        reasons.append("date taken from metadata_modified")

    return {
        "score": score,
        "release_date": release_date,
        "date_source": date_source,
        "dataset": dataset,
        "reasons": reasons,
    }


def find_best_dataset_for_product(product_key: str) -> Dict[str, Any]:
    """
    Search data.gov.uk and return the best candidate dataset.

    Summary:
        1. Try several searches.
        2. Remove not so good results.
        3. Score each remaining good results.
        4. Pick the newest high-scoring result.
    """
    settings = PRODUCTS[product_key]
    candidates_by_id: Dict[str, Dict[str, Any]] = {}

    for query in settings["queries"]:
        print(f"Searching data.gov.uk with q={query!r}")

        results = call_package_search(query, rows=100)
        print(f"  Results returned: {len(results)}")

        for dataset in results:
            dataset_id = str(dataset.get("id", ""))

            if not dataset_id:
                continue

            scored = score_dataset(dataset, product_key, settings)

            if scored is None:
                continue

            existing = candidates_by_id.get(dataset_id)

            if existing is None or scored["score"] > existing["score"]:
                candidates_by_id[dataset_id] = scored

    candidates = list(candidates_by_id.values())

    if not candidates:
        raise RuntimeError(
            f"No usable CSV dataset found for {product_key}. "
            f"Try printing raw package_search results and relaxing filters."
        )

    # Sort by:
    #   1. release/update date, newest first
    #   2. score, highest first
    candidates.sort(
        key=lambda row: (
            row["release_date"],
            row["score"],
        ),
        reverse=True,
    )

    print()
    print(f"Candidate datasets for {product_key}:")
    for number, candidate in enumerate(candidates[:10], start=1):
        dataset = candidate["dataset"]
        print(
            f"{number}. {dataset.get('title')} | "
            f"date={candidate['release_date']} ({candidate['date_source']}) | "
            f"score={candidate['score']} | "
            f"publisher={publisher_name(dataset)}"
        )

    print()

    best = candidates[0]
    dataset = best["dataset"]

    print(f"Selected {product_key}: {dataset.get('title')}")
    print(f"Reason(s): {', '.join(best['reasons'])}")
    print()

    return dataset


# =============================================================================
# 5. Downloading
# =============================================================================

def download_file(url: str, output_path: Path) -> None:
    """
    Download the file in chunks because they can be quite large

    """
    output_path.parent.mkdir(parents=True, exist_ok=True)
    temporary_path = output_path.with_suffix(output_path.suffix + ".part")

    with requests.get(url, stream=True, timeout=(30, 300)) as response:
        response.raise_for_status()

        total_bytes = int(response.headers.get("Content-Length", 0))
        downloaded_bytes = 0

        if total_bytes:
            print(f"File size: {total_bytes / (1024 * 1024):,.1f} MB")
        else:
            print("File size: unknown")

        with temporary_path.open("wb") as file:
            for chunk in response.iter_content(chunk_size=1024 * 1024):
                if not chunk:
                    continue

                file.write(chunk)
                downloaded_bytes += len(chunk)

                downloaded_mb = downloaded_bytes / (1024 * 1024)

                if total_bytes:
                    percent = downloaded_bytes / total_bytes * 100
                    print(f"\rDownloaded {downloaded_mb:,.1f} MB ({percent:,.1f}%)", end="")
                else:
                    print(f"\rDownloaded {downloaded_mb:,.1f} MB", end="")

    print()
    temporary_path.replace(output_path)


def download_product(product_key: str) -> None:
    """
    Find and download one product.
    """
    print("=" * 80)
    print(f"Product: {product_key} - {PRODUCTS[product_key]['friendly_name']}")
    print("=" * 80)

    dataset = find_best_dataset_for_product(product_key)
    csv_resource = choose_csv_resource(dataset)

    title = str(dataset.get("title", product_key))
    csv_url = csv_resource["url"]

    print("Selected CSV resource:")
    print(f"  Resource name: {csv_resource.get('name')}")
    print(f"  Resource format: {csv_resource.get('format')}")
    print(f"  URL: {csv_url}")
    print()

    filename = safe_filename(f"{product_key}_{title}") + ".csv"
    output_path = OUTPUT_FOLDER / filename

    print(f"Saving to: {output_path}")
    download_file(csv_url, output_path)

    print(f"Finished downloading {product_key}.")
    print(f"Saved here: {output_path.resolve()}")
    print()


def main() -> None:
    """
    Run all requested downloads.
    """
    OUTPUT_FOLDER.mkdir(parents=True, exist_ok=True)

    for product_key in PRODUCTS_TO_DOWNLOAD:
        if product_key not in PRODUCTS:
            raise ValueError(f"Unknown product {product_key}. Choose from: {list(PRODUCTS)}")

        download_product(product_key)

    print("All requested downloads complete.")


def run_safely() -> None:
    """
    Run and print the real error if something goes wrong.

    Useful in Jupyter notebooks.
    """
    try:
        main()
    except Exception:
        print()
        print("Something went wrong. Here is the full error:")
        print()
        traceback.print_exc()

## Run the downloader

For a safe test, set `PRODUCTS_TO_DOWNLOAD = ["NSPL"]` before running.


In [2]:
run_safely()


Product: NSPL - National Statistics Postcode Lookup
Searching data.gov.uk with q='"National Statistics Postcode Lookup" "Hosted Table"'
  Results returned: 2
Searching data.gov.uk with q='"National Statistics Postcode Lookup UK"'
  Results returned: 1
Searching data.gov.uk with q='NSPL "National Statistics Postcode Lookup"'
  Results returned: 100

Candidate datasets for NSPL:
1. National Statistics Postcode Lookup UK | date=(2026, 6, 22) (metadata_modified) | score=80 | publisher=london borough of camden
2. National Statistics Postcode Lookup Camden | date=(2026, 6, 22) (metadata_modified) | score=80 | publisher=london borough of camden
3. National Statistics Postcode Lookup (May 2026) for the UK (Hosted Table) | date=(2026, 5, 1) (title_month_year) | score=135 | publisher=office for national statistics
4. National Statistics Postcode Lookup (February 2026) for the UK (Hosted Table) | date=(2026, 2, 1) (title_month_year) | score=135 | publisher=office for national statistics

Selected